# RAG
Retrieval Augmented Generation (검색증강생성)






In [1]:
# 사전학습된 모델은 이미 많은 데이터를 통해 학습한 상태이긴 하나..
# 개인 DB 나 회사내 문서 와 같이 'private 한 데이터' 들에는 접근할수 없다
# 그래서 RAG 를 사용한다!


In [2]:
# RAG 는 특정 라이브러리나 프레임워크 이름이 아니라
# 위와 같은 작업을 하는 '기법'을 일반적으로 통칭하는 용어


# RAG 를 수행하는 방법은 굉~장히 많고 다양.




# 어떤 방식으로 RAG 를 구현할른지는

# - 우리가 얼마나 많은 문서들을 가지고 있는지
# - 우리가 얼마나 많은 비용으로 운영할지 (어떤 모델, 가용한 token 개수등..)

# 등에 따라 결정될 문제다.


In [3]:
# 1. Retrieval 단계
# private 으로부터 제공된 data 를 사용하거나 탐색함으로써
# language model 의 능력을 더 '확장(augment)'


# 2. Augmented Generation
# Model 로 하여금 '우리가 보낸 문서 data 만'을 가지고 답변하도록 할수도 있다.
# (경우에 따라, 우리 문서가 더 최신 data 일수도 있기 때문이다)
# 이를 통해 Model 이 과거에 학습한 data 를 참조하지 않게도 할수 있다.


# Retrieval 단계
![](https://miro.medium.com/v2/resize:fit:1100/format:webp/0*jmVYxqojDFn2yoOr.jpg)





In [4]:
# RAG 의 첫번째 단계인 Retrieval 의 일반적인 과정
# - data source 에서 데이터 load
# - 데이터는 split 하면서 transform
# - transform 한 데이터를 embed.
# - embed 된 데이터를 store 에 저장.
# - 검색(질의) 가 입력되면 store 에서 관련 문서들을 retrieve!

# Data Loaders

In [5]:
# 랭체인에서 제공하는 다양한 document loader 들이 있다
# CSV, File Directory, HTML, JSON, Markdown, PDF 등
# ※그 밖에서도 3rd party loader 들도 있다.

## 파일 준비

In [6]:
# 아래와 같이 파일들을 준비합니다


# 출처는  조지오웰의 소설 '1984' Part1 Chapter1
#  http://www.george-orwell.org/1984/0.html


# 너무 길거나, 너무 짧지 않으면 좋습니다
# 파일이 너무 길면 나중에 임베딩 과정에서 비용지출이 발생.


# 다운로드 링크
# https://www.dropbox.com/scl/fi/ppid6hk7bwqxc0xrv65oc/files.zip?rlkey=df5d411n1fnaht2wlwv0o71wt&st=ddd52gw3&dl=1


In [7]:
base_path = r'F:\KDT2508\dataset\files'

# import

In [8]:
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.runnables.passthrough import RunnablePassthrough
import os 

In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
llm = ChatOpenAI(temperature=0.1)

## TextLoader

In [11]:
from langchain_community.document_loaders.text import TextLoader

In [12]:
loader = TextLoader(os.path.join(base_path, 'chapter_one.txt'))

In [13]:
docs = loader.load()
len(docs)

1

In [14]:
doc = docs[0]

In [15]:
doc.page_content[:300] # .page_content에 실제 데이터

'Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of'

# PyPDFLoader

In [16]:
from langchain_community.document_loaders.pdf import PyPDFLoader

In [17]:
loader=PyPDFLoader(os.path.join(base_path,'chapter_one.pdf'))
loader.load()

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-01-30T23:19:00+09:00', 'author': 'Yeonchul Sung', 'moddate': '2025-01-30T23:19:00+09:00', 'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Part 1, Chapter 1 \n \n \nPart One \n \n \n1 \nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his \nchin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through \nthe glass doors of Victory Mansions, though not quickly enough to prevent a swirl of \ngritty dust from entering along with him. \n \nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured \nposter, too large for indoor display, had been tacked to the wall. It depicted simply an \nenormous face, more than a metre wide: the face of a man of about forty-five, with a \nheavy black moustache and ruggedly handsome 

# UnstructuredFileLoader

In [18]:
# 서로 다른 타입의 문서를 읽어오기 위해 각각의 DataLoader 를 사용하기 보다
# UnstructuredFileLoader 라는 것도 사용해볼수 있다. -> 꽤 다양한 포맷의 파일을 읽어올 수 있다

In [19]:
from langchain_community.document_loaders.unstructured import UnstructuredFileLoader

In [20]:
loader = UnstructuredFileLoader(os.path.join(base_path, 'chapter_one.pdf'))
loader.load()

C:\Users\msi\AppData\Local\Temp\ipykernel_12224\2677622240.py:1: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader(os.path.join(base_path, 'chapter_one.pdf'))


[Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.pdf'}, page_content="Part 1, Chapter 1\n\nPart One\n\n1\n\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his\n\nchin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through\n\nthe glass doors of Victory Mansions, though not quickly enough to prevent a swirl of\n\ngritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured\n\nposter, too large for indoor display, had been tacked to the wall. It depicted simply an\n\nenormous face, more than a metre wide: the face of a man of about forty-five, with a\n\nheavy black moustache and ruggedly handsome features. Winston made for the stairs. It\n\nwas no use trying the lift. Even at the best of times it was seldom working, and at\n\npresent the electric current was cut off during daylight hours. It was part of the economy\n\ndrive in pr

In [21]:
loader = UnstructuredFileLoader(os.path.join(base_path, 'chapter_one.docx'))
loader.load()

[Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.docx'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. T

# Spliter

## data 를 split해야하는 이유!


In [22]:
# loader.load() 의 리턴값을 보면 'Document로 이루어진 list' 다.
# 지금의 경우는 전체 챕터가 '하나의 Document' 에 들어가 있다.

len(loader.load())

1

In [23]:
# 특정 질문에 답해야 하기 위해서, 필요한 '파일의 일부분' 만들 전달해야 할 수도 있다.

#  그래서 문서를 쪼개두어야(split) 한다

# 가령: "Ministry of peace" 를 찾고자 한다면.
# 해당 키워드가 있는 문서(들)만 모델에 넘겨주면 된다.

# 작은 조각들로 쪼개어 두면 필요한 것들을 찾기가 용이해진다.
#  - prompt 도 짧아질거다 (적은 token 사용, 적은 비용.)

# split 하는 방법은 다양하다.

In [24]:
"""
TextSplitter 계층도

BaseDocumentTransformer --> TextSplitter --> <name>TextSplitter  # Example: CharacterTextSplitter
                                             RecursiveCharacterTextSplitter -->  <name>TextSplitter

https://python.langchain.com/api_reference/text_splitters/index.html

"""
None


## RecursiveCharacterTextSpliter

In [25]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

In [26]:
splitter = RecursiveCharacterTextSplitter()

In [27]:
# RecursiveCharacterTextSplitter 는 파일을 split 해주는데
# 문장의 끝이나, 문단의 끝부분마다 끊어준다.
# 문장 중간을 끊지는 않는다.  최대한 문장 중간에서 split 되지 않도록 하려 한다.
# 문장 중간에 짤림으로 의미있는 문장들을 잃고 싶지 않다.

# ↓ splitter 사용방법은 두가지 가 있다.

In [28]:
docs = loader.load() # -> list[Document]

In [29]:
# 방법1 
documents = splitter.split_documents(docs)
print(len(documents), '개의 documents')
documents

11 개의 documents


[Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.docx'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. T

In [30]:
# 방법2
documents = loader.load_and_split(text_splitter=splitter) #-> 쪼개진 list[document]
print(len(documents), '개 documents')
documents

11 개 documents


[Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.docx'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. T

In [31]:
# 첫번째 document
print(documents[0].page_content)

Part 1, Chapter 1

Part One


1
It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.

The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above his righ

In [32]:
# 문장의 문당 구조를 유지하면서 split!

## chunk_size

In [33]:
# 좀 더 작은 Document 를 만들 필요가 있다.
# 모델의 Context Window 가 크지 않은 경우라든지..
# chunk_size= 값으로 조정해보자

In [34]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, # 얼마나 큰 덩어리로 나눌지 지정.  (아까보다 굉장히 잘개 쪼개질거다)
                    # chunk_size 의 단위는 splitter 마다 다르다.
                    # CharacterTextSplitter 의 경우 chunk_size 는 문자개수
)


documents = loader.load_and_split(text_splitter=splitter)
print(len(documents), '개 documents')

documents[:5]

3498 개 documents


[Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.docx'}, page_content='Part 1, Chapter 1\n\nPart One'),
 Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.docx'}, page_content='1'),
 Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.docx'}, page_content='It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors'),
 Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.docx'}, page_content='was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of'),
 Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.docx'}, page_content='cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzz

In [35]:
# ↑ 문제점: 문단의 중간이 잘려버렸다 -> 문장의 의미가 파괴된다.

# 작은 덩어리이면서 문장의 중간을 잘라먹지 않는 방법은?
# chunk_overlap=
#    split 할때 앞 조각의 일부를 가져와서 연결해준다.
#    Document 간의 겹치는 부분 생길수 있다.

In [36]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)


documents = loader.load_and_split(text_splitter=splitter)
print(len(documents), '개 documents')

for document in documents[10:15]:
    print('🦄', document.page_content)

250 개 documents
🦄 move. BIG BROTHER IS WATCHING YOU, the caption beneath it ran.
🦄 Inside the flat a fruity voice was reading out a list of figures which had something to do with the production of pig-iron. The voice came from an oblong metal plaque like a dulled mirror which
🦄 an oblong metal plaque like a dulled mirror which formed part of the surface of the right-hand wall. Winston turned a switch and the voice sank somewhat, though the words were still distinguishable.
🦄 though the words were still distinguishable. The instrument (the telescreen, it was called) could be dimmed, but there was no way of shutting it off completely. He moved over to the window: a
🦄 it off completely. He moved over to the window: a smallish, frail figure, the meagreness of his body merely emphasized by the blue overalls which were the uniform of the party. His hair was very


In [37]:
# ↑ Document 간에 겹치는 부분이 있다.
# 앞 Document 의 뒷부분을 가져다가 다음 Document 의 앞에 넣었다.
# 이렇게 하므로 문장의 (의미적) 구조를 크게 해치지 않도록 split 했다.

## CharacterTextSplitter

In [38]:
from langchain_text_splitters.character import CharacterTextSplitter

In [39]:
# CharacterTextSplitter 도 동작방식은 비슷하다
# separator=  : 특정 문자열 찾은 다음 이를 기준으로 분할한다.

splitter = CharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separator = '\n' # 줄바꿈 단위로 split
)


documents = loader.load_and_split(text_splitter=splitter)
print(len(documents), '개 documents')

for document in documents[10:15]:
    print('🦄', document.page_content)

Created a chunk of size 963, which is longer than the specified 600
Created a chunk of size 774, which is longer than the specified 600
Created a chunk of size 954, which is longer than the specified 600
Created a chunk of size 922, which is longer than the specified 600
Created a chunk of size 881, which is longer than the specified 600
Created a chunk of size 821, which is longer than the specified 600
Created a chunk of size 700, which is longer than the specified 600
Created a chunk of size 745, which is longer than the specified 600
Created a chunk of size 735, which is longer than the specified 600
Created a chunk of size 671, which is longer than the specified 600
Created a chunk of size 991, which is longer than the specified 600
Created a chunk of size 990, which is longer than the specified 600
Created a chunk of size 1289, which is longer than the specified 600
Created a chunk of size 1605, which is longer than the specified 600
Created a chunk of size 1900, which is longer 

46 개 documents
🦄 Winston turned round abruptly. He had set his features into the expression of quiet optimism which it was advisable to wear when facing the telescreen. He crossed the room into the tiny kitchen. By leaving the Ministry at this time of day he had sacrificed his lunch in the canteen, and he was aware that there was no food in the kitchen except a hunk of dark-coloured bread which had got to be saved for tomorrow's breakfast. He took down from the shelf a bottle of colourless liquid with a plain white label marked VICTORY GIN. It gave off a sickly, oily smell, as of Chinese ricespirit. Winston poured out nearly a teacupful, nerved himself for a shock, and gulped it down like a dose of medicine.
🦄 Instantly his face turned scarlet and the water ran out of his eyes. The stuff was like nitric acid, and moreover, in swallowing it one had the sensation of being hit on the back of the head with a rubber club. The next moment, however, the burning in his belly died down and the 

## length_function=

In [40]:
# splitter 에 lenth 를 계산하는 함수를 제공해줄수 있다.
#  length_function=   
#    기본적으론 파이썬의 len() 을 사용한다 (디폴트) 

splitter = CharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separator = '\n', # 줄바꿈 단위로 split
    length_function=len, # default 값이 chunk 카운트 함수
)


In [41]:
# 디폴트로 len() 함수가 동작함. CharacterTextSplitter 에선 '글자의 개수'를 chunk 카운트 함.
# 그러나 LLM 에서 말하는 token 은 문자(letter) 와는 다르다.
# 어떤 경우에는 문자 두개, 혹은 세개...  가 한개의 token 으로 카운트 된다.


# TikToken

## OpenAI Tokenizer 예시

In [42]:
# OpenAI 에서의 token 예시
# https://platform.openai.com/tokenizer
# ↓ model 의 관점에서, 몇개의 token 을 사용하는지 확인해 볼수 있다.

## from_titkoen_encoder()

In [43]:
splitter = CharacterTextSplitter.from_tiktoken_encoder( #단위가 글자에서 token으로 바
    chunk_size=600, 
    chunk_overlap=100,
    separator = '\n',
)

# 이제 '모델'이 텍스트를 세는 방법과 'splitter'가 텍스트를 세는 방법이 일치 하게 되었다.
# model 에는 입력 limit 이 있기 때문에 (context window), 원하는 텍스트들을 모두 한번에 입력할 수는 없다.
# 그래서 우리 텍스트를 길이 계산할때 model 과 같은 방법으로 계산하는게 더 좋다.



# Vectors

## Embedding 과 Vector

![](https://miro.medium.com/v2/resize:fit:2000/1*SYiW1MUZul1NvL1kc1RxwQ.png)



## word2vec 예시

https://turbomaze.github.io/word2vecjson/




# Vector Store

## OpenAIEmbeddings

In [44]:
from langchain_openai.embeddings.base import OpenAIEmbeddings

In [45]:
embedder = OpenAIEmbeddings()

In [46]:
embedder.model

# text-embedding-ada-002'   
#  https://platform.openai.com/docs/models/text-embedding-ada-002
#  1M token 당 $0.1

'text-embedding-ada-002'

In [47]:
# OpenAIEmbeddings 를 통해
#  embed_documents()  <- 문서를 embed 하는것 뿐만 아니라
#  embed_query()      <- query 도 embed 하는 것이 가능하다.


In [48]:
vector = embedder.embed_query('Hi')
print(len(vector))
print(vector)

1536
[-0.03629858046770096, -0.007224537897855043, -0.03371885418891907, -0.02866363152861595, -0.02686564065515995, 0.03460482135415077, -0.012318846769630909, -0.007752209436148405, 0.0019380523590371013, -0.0027018729597330093, 0.024781012907624245, -0.002477124100551009, -0.00573272630572319, -0.002905449829995632, 0.006677323020994663, -0.00303248199634254, 0.033849142491817474, -0.001503212028183043, 0.02109382674098015, -0.008996471762657166, -0.02171921543776989, 0.01038405206054449, 0.006244111340492964, 0.007081219926476479, -0.012312332168221474, 0.0008998099947348237, 0.005876044277101755, -0.009888952597975731, -0.0030731973238289356, -0.024572549387812614, 0.010742347687482834, -0.01381065882742405, -0.024429231882095337, -0.01411032397300005, 0.0024347801227122545, -0.018878910690546036, 0.0005618723225779831, -0.011270018294453621, 0.018110202625393867, -0.009967125952243805, 0.01302892342209816, -0.011328648775815964, -0.009133275598287582, -0.009654432535171509, -0.02

In [49]:
vector = embedder.embed_documents([
    'hi',
    'how are you',
    'great to see you'
]) # -> list[list[flaot]] 2차원 리스트 return 

In [50]:
len(vector)

3

In [51]:
for v in vector:
    print(len(v))

1536
1536
1536


In [52]:
# 코드를 실행할때마다 '매번' 문서 embedding 을 반복해서 수행하는건 매우 비효율적이다
#  => 시간 소요 + 또한 비용 지출

# 대신! 그 embeded 된 결과들을 '저장'해 줄겁니다.
# LangChain 은 embedding 한것들을 캐싱하는 기능을 제공해준다

# '동일 Document'는 가급적 한번만 embedding 해주는게 좋다.


## Vector Store란?

In [53]:
# 일단 벡터를 만들고 나서, 그것들을 캐시해주고, vector store 에 넣어주면,
# 우리가 '검색'을 할수 있다.
# 그리하여, '관련있는 문서'들만 찾아낼수 있게 되는 거다

# 랭체인은 다양한 vector store 를 제공한다,  어떤거는 cloud 형태이고, 어떤건 유료이기도 하다.

# 우리는 예제에서 무료로 사용할수 있고 로컬로 저장되는 Chroma 라는 것을 사용해볼겁니다

# 나중에 FAISS라는 메모리기반 vector stor도 사용해볼거고...

# 클라우드기반의 vector store인 pinecone 도 사용해 보자.

## Chroma Vector Store

In [54]:
from langchain_community.vectorstores.chroma import Chroma

In [55]:
# ↓이 ChromaDB 에 'split 된 문서' 와 'OpenAI embedding model' 을 전달해야 한다

# OpenAIEmbeddings 의 옵션에 model= 이 있다. 여기에 원하는 모델 지정가능 (지정안하면 default 동작)

# ★ embedding 모델을 사용하는것도 비용이 발생한다!

In [56]:
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

loader = UnstructuredFileLoader(os.path.join(base_path, 'chapter_one.docx'))

docs = loader.load_and_split(text_splitter=splitter)

# embedding 모델 준비
embeddings = OpenAIEmbeddings()

In [57]:
vectorstore = Chroma.from_documents(docs, embeddings)
# ↑ ★★ 이 코드 실행하면 비용지출 발생함.
#        문서의 크기가 클수록 당연히 비례해서 비용 발생
#        우리 예제에서는 작은 파일을 사용하는 것이니 매우 적은 비용이 발생할 것이다.


In [58]:
# 이제 vectorstore 를 사용하여 '유사도' 검색을 수행해보자
# 우리 Document 들이 벡터와 되었고, 이제 벡터공간에 대한 검색을 해볼수 있다.

result = vectorstore.similarity_search('where does Winston live?') # list return
print(len(result), '개 documents ')
result

4 개 documents 


[Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.docx'}, page_content='Part 1, Chapter 1\nPart One\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat 

In [59]:
result[0].page_content

'Part 1, Chapter 1\nPart One\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above his rig

In [60]:
result[1].page_content

"The Ministry of Love was the really frightening one. There were no windows in it at all. Winston had never been inside the Ministry of Love, nor within half a kilometre of it. It was a place impossible to enter except on official business, and then only by penetrating through a maze of barbed-wire entanglements, steel doors, and hidden machine-gun nests. Even the streets leading up to its outer barriers were roamed by gorilla-faced guards in black uniforms, armed with jointed truncheons.\nWinston turned round abruptly. He had set his features into the expression of quiet optimism which it was advisable to wear when facing the telescreen. He crossed the room into the tiny kitchen. By leaving the Ministry at this time of day he had sacrificed his lunch in the canteen, and he was aware that there was no food in the kitchen except a hunk of dark-coloured bread which had got to be saved for tomorrow's breakfast. He took down from the shelf a bottle of colourless liquid with a plain white l

## Embedding cache

In [61]:
# 다시 실행하면 임베딩 결과는 다 사라진다. 재실행하면 다시 재계산 발생 (비용발생!)
# 그래서 embedding 을 캐싱해주자

In [62]:
from langchain_classic.embeddings import CacheBackedEmbeddings

In [63]:
from langchain_classic.storage import LocalFileStore

In [64]:
# 캐시 경로 지정. 여기에 embedding 이 저장
cache_dir = LocalFileStore(os.path.join(base_path, '.cache')) # 플젝때는 보통 gitignore

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings, # 임베딩 모델
    cache_dir,
)

vectorstore = Chroma.from_documents(docs, cached_embeddings)

results = vectorstore.similarity_search("where does winston live")
results

F:\KDT2508\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


[Document(metadata={'source': 'F:\\KDT2508\\dataset\\files\\chapter_one.docx'}, page_content="The Ministry of Love was the really frightening one. There were no windows in it at all. Winston had never been inside the Ministry of Love, nor within half a kilometre of it. It was a place impossible to enter except on official business, and then only by penetrating through a maze of barbed-wire entanglements, steel doors, and hidden machine-gun nests. Even the streets leading up to its outer barriers were roamed by gorilla-faced guards in black uniforms, armed with jointed truncheons.\nWinston turned round abruptly. He had set his features into the expression of quiet optimism which it was advisable to wear when facing the telescreen. He crossed the room into the tiny kitchen. By leaving the Ministry at this time of day he had sacrificed his lunch in the canteen, and he was aware that there was no food in the kitchen except a hunk of dark-coloured bread which had got to be saved for tomorro

In [65]:
# 위코드를 실행하여 우리가 또 파일 embedding 작업을 할때는,
# 1.첫번째로!
#   캐시에 embeddings 가 이미 존재하는지 확인할거다.
# 2.만약 없다면!
#    vector store(Chroma.from_documents) 를 호출할 때
#   문서들(docs) 과 함께 OpenAIEmbeddings 를 사용할거다.
#

# Stuff Document Chain

![](https://publish-01.obsidian.md/access/84bff78007c07f71220692232b80322e/Study/LangChain/Reference/RAG%20Reference/Stufff%20Chain%20Process.jpg)


In [ ]:
# Studd Document Chain 을 가장 단순한 처리 방식
# 여러개의 문서(document)를 하나로 합쳐서 (stuffing) 프롬포트에 넣어 LLM이 전
# 문서의 양이 LLM의 context window 보다 작다면 매우 빠르고 효율적 

In [66]:
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [67]:
llm = ChatOpenAI(temperature=0.1)

In [73]:
# 데이터 준비
docs = [
    Document(page_content='사과는 빨간색 과일입니다'),
    Document(page_content='바나나는 노란색 과일입니다.')
]

In [77]:
# prompt template
prompt = ChatPromptTemplate.from_template('''
    다음 제공된 문맥(context)만을 사용하여 질문에 답하세요. 모르면 모른다해
    {context}

    질문 : {question}
''')

In [79]:
# 문서를 하나로 합치는 함수를 정의
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

    
chain = (
    {
        'context': lambda x : format_docs(x['document']),
        'question' : lambda x : x['question']
    } 
    | prompt 
    | llm
)

# 호출 
response = chain.invoke({
    'document' : docs,
    'question' : '복숭아는 무슨색?'
})

print(response.content)

복숭아는 분홍색 과일입니다.


# Map-Reduce chain

![](https://publish-01.obsidian.md/access/84bff78007c07f71220692232b80322e/Study/LangChain/Reference/RAG%20Reference/Map%20Reduce%20Chain%20Process.jpg)




In [86]:
# 1. Map 단계 prompt
from langchain_core.output_parsers import StrOutputParser


map_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 문서를 요약하는 AI입니다.'),
    ('human', '다음 문서를 핵심만 요약하세요 : \n\n {documents}'),
])

map_chain = map_prompt | llm | StrOutputParser()

In [87]:
# --------------------
# 4. 테스트용 문서 준비
# --------------------
documents = [
    Document(page_content="LangChain은 LLM 애플리케이션을 구축하기 위한 프레임워크입니다."),
    Document(page_content="LCEL은 LangChain에서 체인을 선언적으로 구성할 수 있도록 도와줍니다."),
    Document(page_content="Map-Reduce 체인은 대규모 문서를 효율적으로 요약하는 데 유용합니다.")
]


In [88]:
# 동작 확인
map_chain.invoke({'documents' : documents[0].page_content})

'LangChain은 LLM 애플리케이션을 위한 프레임워크이다.'

In [90]:
# Reduce 단계 prompt

reduce_prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 여러 요약을 하나의 요약으로 통합하는 AI이다.'),
    ('human', '다음 요약들을 하나의 간결한 요약으로 통합하세요 : \n\n{summaries}')
])

reduce_chain = reduce_prompt | llm | StrOutputParser()

In [91]:
# 동작 확인
reduce_chain.invoke({
    "summaries" : '\n\n'.join(document.page_content for document in documents)
})

'LangChain은 LLM 애플리케이션을 구축하고, LCEL을 사용하여 체인을 선언적으로 구성하며, Map-Reduce 체인은 대규모 문서를 효율적으로 요약하는 데 활용됩니다.'

In [94]:
from langchain_core.runnables import RunnableLambda
# Map-Reduce 체인

map_reduce_chain=(
    # Map 단계
    RunnableLambda(
        lambda docs : [map_chain.invoke({'documents' : doc.page_content}) for doc in docs]
    )
    # Reduce 단계
    | RunnableLambda(
        lambda summaries : reduce_chain.invoke({'summaries' : '\n\n'.join(summaries)})
    )
)

map_reduce_chain.invoke(documents)

'LangChain은 LLM 애플리케이션을 위한 프레임워크로, LCEL을 사용하여 선언적인 방식으로 체인을 구성하며, Map-Reduce 체인은 대규모 문서를 효율적으로 요약하는 데 유용하다.'

# Map-Rerank Chain

![](https://publish-01.obsidian.md/access/84bff78007c07f71220692232b80322e/Study/LangChain/Reference/RAG%20Reference/Map%20Re-rank%20Process.png)


In [ ]:
# Map-rerank 흐름

# 1. Map 단계 
#   각 Document를 독립적으로 LLM에 전달
#   답변 + 관련성 점수 생성

# 2. Rerank 단계
#   점수를 기준으로 가장 좋은 답변을 선택

In [95]:
# Map 단계

map_prompt = ChatPromptTemplate.from_template(
    """
    너는 질문-응답 평가 모델이다.

    질문:
    {question}

    문서:
    {context}

    위 문서를 기반으로 질문에 답하라.
    그리고 답변의 질문 관련성을 0~100 사이의 점수로 평가하라.

    반드시 아래 형식으로만 출력하라:
    score: <점수>
    answer: <답변>
    """
)

In [96]:
output_parser = StrOutputParser()

In [97]:
map_chain = map_prompt | llm | output_parser

In [98]:
documents = [
    Document(page_content="LangChain은 LLM 애플리케이션 프레임워크이다."),
    Document(page_content="LCEL은 LangChain의 선언형 체인 문법이다."),
    Document(page_content="Map-Rerank는 문서별 점수를 비교하는 체인이다.")
]

question = "LangChain의 LCEL은 무엇인가?"



In [99]:
map_result = [
    map_chain.invoke({
        'question' : question,
        'context' : doc.page_content,
    })
    for doc in documents
]

map_result

['score: 90\nanswer: LCEL은 LangChain Embedding Layer의 약자로, LangChain 프레임워크에서 사용되는 임베딩 레이어를 의미한다.',
 'score: 100\nanswer: LCEL은 LangChain의 선언형 체인 문법입니다.',
 'score: 0\nanswer: 해당 문서에는 LangChain의 LCEL에 대한 정보가 없습니다.']

In [132]:
# Rerank 단계
import re

def parse_result(text:str):
    score_match = re.search(r'score:\s*(\d+)', text)
    answer_match = re.search(r'answer:\s*(.*)', text, re.DOTALL)  # re.DOTALL는 정규표현식 ".*"에서 줄바꿈 (\n)도 포함해서 매칭 

    return{
        'score' : int(score_match.group(1)),
        'answer' : answer_match.group(1).strip()
    }

In [133]:
# 동작 확인
parse_result(r'score: 100\nanswer: LCEL은 LangChain의 선언형 체인 문법입니다.')

{'score': 100, 'answer': 'LCEL은 LangChain의 선언형 체인 문법입니다.'}

In [134]:
parsed_result = [parse_result(r) for r in map_result]

In [137]:
best_result = max(parsed_result, key=lambda x :x['score'])

best_result

{'score': 100, 'answer': 'LCEL은 LangChain의 선언형 체인 문법입니다.'}

# Refine Document Chain

![](https://publish-01.obsidian.md/access/84bff78007c07f71220692232b80322e/Study/LangChain/Reference/RAG%20Reference/Refine%20Chain%20Process.jpg)


In [ ]:
# Refine Document Chain
#  - 여러개의 Dcument 를 순차적으로 처리
#  - 첫번째 문서로부터 초기 답변 생성
#  - 이후 문서들을 하나씩 읽으면서 기존 답변을 개선 (refine)

In [138]:
documents = [
    Document(page_content="LangChain은 LLM 애플리케이션을 구축하기 위한 프레임워크이다."),
    Document(page_content="LCEL은 체인을 선언적으로 구성할 수 있게 해준다."),
    Document(page_content="Refine 체인은 문서를 순차적으로 처리하며 답변을 개선한다.")
]


In [139]:
# 초기 답변생성용 프롬포트
from langchain_core.prompts import PromptTemplate

initial_prompt = PromptTemplate.from_template(
    '''
    다음 문서를 기반으로 질문에 대한 답변을 작성하세요.

    문서:
    {context}

    질문:
    {question}

    답변:

    '''
)

In [140]:
# 초기 chain (첫번째 문서용)

initial_chain = initial_prompt | llm | output_parser

In [141]:
# 테스트
initial_chain.invoke({
    'context' : documents[0].page_content,
    'question' : 'LangChain 에서 Refine chain이 무엇인지 설명해줘'
})

'LangChain에서 Refine chain은 데이터를 정제하고 가공하는 과정을 의미합니다. 이 과정은 데이터를 정확하고 일관된 형식으로 변환하고, 노이즈를 제거하며, 필요한 정보를 추출하는 등의 작업을 포함합니다. Refine chain은 LLM 애플리케이션을 구축하는 과정에서 데이터의 품질을 향상시키고, 분석 및 학습에 활용할 수 있는 최적의 형태로 데이터를 가공하는 역할을 합니다.'

In [142]:
# Refine 프롬포트
refine_prompt = PromptTemplate.from_template(
    '''
     기존 답변:
    {existing_answer}

    아래 문서를 참고하여 기존 답변을 개선하세요.
    만약 도움이 되지 않으면 기존 답변을 그대로 유지하세요.

    문서:
    {context}

    질문:
    {question}

    개선된 답변:
    '''
)

In [143]:
# refine 체인, 두번째 문서부터!
refine_chain = refine_prompt | llm | output_parser

In [144]:
# refine 로직 
def refine_documents(inputs): # inputs <- documents, question을 담음
    docs = inputs['documents']
    question = inputs['question']

    # 첫 문서로 초기 답변 
    answer = initial_chain.invoke({
        'context' : docs[0].page_content,
        'question' : question,
    })

    # 나머지 문서들로 순차적 refine
    for doc in docs[1:]:
        answwer = refine_chain.invoke({
            'existing_answer':answer,
            'context':doc.page_content,
            'question':question 
        })

    return answer

refine_documents_chain = RunnableLambda(refine_documents)

refine_documents_chain.invoke({
    'documents' : documents,
    'question' : "Langchain 에서 Refinechain 이 무엇인지 설명해줘"
})

'LangChain에서 RefineChain은 데이터 정제 및 가공을 위한 서브 프레임워크입니다. RefineChain은 LangChain의 기능을 보완하여 데이터를 정확하고 효율적으로 처리할 수 있도록 도와줍니다. 데이터의 품질을 향상시키고 분석에 활용할 수 있는 형태로 변환하는 작업을 수행합니다. RefineChain은 LangChain의 성능과 유용성을 높이는 역할을 합니다.'